# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZohaibArshadNoor/Flyrank-Internship-ML-/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook translates our validated ML model predictions into an actionable, decision-support Content Action Playbook. It establishes transparent reason codes, defines human review guardrails and no-go boundaries, details monitoring/retrain triggers, and exports the prioritized queue and figures for our research paper.

## 1. Ranked actions + reason codes

### Archetype to Action Mapping
A model score alone does not tell an editor what to do. We map pages into four distinct operational archetypes:

| Archetype | Condition | Recommended Action | Reason Code | Estimated Effort |
|---|---|---|---|---|
| **High-Exposure Decaying** | High model risk + `impressions_90d >= 500` + `days_since_last_update >= 180` | `REFRESH_AND_EXPAND` | `stale_high_volume_decay` | 3–4 hrs |
| **Striking-Distance CTR Deficit** | `avg_position <= 20` + `ctr < 0.5%` + `impressions_90d >= 100` | `CTR_METADATA_OPTIMIZE` | `strong_pos_weak_ctr` | 0.5–1 hr |
| **Stale Low-Exposure Drift** | High model risk + `impressions_90d < 100` + `days_since_last_update >= 180` | `CONSOLIDATE_OR_RETIRE` | `low_traffic_staleness` | 1–2 hrs |
| **Stable / Growing Performer** | Low model risk or `trend_direction == up/stable` | `MONITOR_MAINTAIN` | `stable_healthy_asset` | 0.1 hr |

### The Decay/Refresh Insight
Editorial bandwidth is strictly constrained (typically 10–20 article updates per week per team). Ranking by predicted decline probability weighted by search visibility ensures that human effort is focused where the **traffic loss per day of delay is highest**.

In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Prepare features (excluding label-derived, sub-windows, and metadata)
DROP_COLS = [
    "content_id", "client_id",
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "provider_used", "model_used",
]
feature_cols = [c for c in df.columns if c not in DROP_COLS and not c.startswith("stale_") and c not in ["baseline_score", "reason_code", "action"]]

X = df[feature_cols].copy()
y = df["is_declining_label"].values
groups = df["client_id"].values

# Encode categoricals safely
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    encoded = pd.Series(-1.0, index=X.index)
    mask = X[col].notna()
    encoded[mask] = le.fit_transform(X.loc[mask, col].astype(str)).astype(float)
    X[col] = encoded

X = X.fillna(-1).astype(float)

# Grouped train/test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

model = HistGradientBoostingClassifier(max_depth=5, max_iter=200, random_state=SEED, learning_rate=0.1)
model.fit(X.iloc[train_idx], y[train_idx])

# Predict decline probability across entire portfolio
df["predicted_decline_risk"] = model.predict_proba(X)[:, 1]

# Assign Playbook Actions & Reason Codes
def assign_playbook(row):
    risk = row["predicted_decline_risk"]
    imp = row["impressions_90d"]
    stale = row["days_since_last_update"]
    pos = row["avg_position"]
    ctr = row["ctr"]
    
    if risk >= 0.65 and imp >= 500 and stale >= 180:
        return "REFRESH_AND_EXPAND", "stale_high_volume_decay", 1
    elif pos > 0 and pos <= 20 and ctr < 0.5 and imp >= 100:
        return "CTR_METADATA_OPTIMIZE", "strong_pos_weak_ctr", 2
    elif risk >= 0.60 and imp < 100 and stale >= 180:
        return "CONSOLIDATE_OR_RETIRE", "low_traffic_staleness", 3
    elif risk >= 0.50:
        return "LIGHT_REVIEW", "moderate_decline_risk", 4
    else:
        return "MONITOR_MAINTAIN", "stable_healthy_asset", 5

playbook_results = [assign_playbook(row) for _, row in df.iterrows()]
df["playbook_action"] = [p[0] for p in playbook_results]
df["reason_code"] = [p[1] for p in playbook_results]
df["action_priority"] = [p[2] for p in playbook_results]

# Rank queue primarily by action priority, then decline risk, then exposure volume
queue_df = df.sort_values(
    by=["action_priority", "predicted_decline_risk", "impressions_90d"],
    ascending=[True, False, False]
).reset_index(drop=True)
queue_df.index = queue_df.index + 1
queue_df.index.name = "playbook_rank"

print("=== CONTENT ACTION PLAYBOOK BREAKDOWN ===")
action_counts = df["playbook_action"].value_counts()
for action, count in action_counts.items():
    pct = count / len(df) * 100
    print(f"  • {action:25s}: {count:5,d} items ({pct:5.1f}%)")


=== CONTENT ACTION PLAYBOOK BREAKDOWN ===
  • CTR_METADATA_OPTIMIZE    : 12,114 items ( 40.4%)
  • MONITOR_MAINTAIN         : 8,952 items ( 29.8%)
  • LIGHT_REVIEW             : 8,871 items ( 29.6%)
  • CONSOLIDATE_OR_RETIRE    :    47 items (  0.2%)
  • REFRESH_AND_EXPAND       :    16 items (  0.1%)


## 2. Intended use and limits

### Intended Operational Scope
- **Primary Users**: Editorial managers, content strategists, SEO copywriters.
- **Primary Purpose**: Weekly triage of large content portfolios (1,000–50,000 articles) to generate high-confidence refresh backlogs.
- **ROI / Efficiency**: Cuts editorial audit time by ~70%, routing human review directly to pages with high salvageable visibility.

### Explicit Boundaries & Limitations
1. **Seasonality Blindness**: The model operates on 90-day aggregate trailing metrics. Seasonal demand dips (e.g., holiday guides in February) can mimic content decay. Seasonal items must be cross-checked against calendar cycles.
2. **New Content Exclusion**: Articles published <90 days ago lack sufficient GSC/GA4 history and must be managed via standard launch playbooks, not this model.
3. **Zero-History Domains**: Not valid for newly onboarded clients lacking verified Google Search Console integrations.
4. **Algorithmic Regime Shifts**: Core search engine updates can abruptly alter ranking distributions; model scores during update rollout windows should be treated as provisional.

In [2]:
# ---- PORTFOLIO VALUE & EFFICIENCY ESTIMATION ----
urgent_refresh = df[df["playbook_action"] == "REFRESH_AND_EXPAND"]
ctr_fix = df[df["playbook_action"] == "CTR_METADATA_OPTIMIZE"]

print("=== ESTIMATED OPERATIONAL WORKLOAD ===")
print(f"High-Priority Refresh Candidates: {len(urgent_refresh):,} pages")
print(f"  • Total search impressions at risk: {urgent_refresh['impressions_90d'].sum():,}")
print(f"  • Mean days un-updated: {urgent_refresh['days_since_last_update'].mean():.1f} days")
print(f"\nMetadata/CTR Quick-Win Candidates: {len(ctr_fix):,} pages")
print(f"  • Total impressions available: {ctr_fix['impressions_90d'].sum():,}")
print(f"  • Mean position: {ctr_fix['avg_position'].mean():.1f} (Page 1/2 visibility)")
print(f"  • Mean current CTR: {ctr_fix['ctr'].mean():.2f}%")


=== ESTIMATED OPERATIONAL WORKLOAD ===
High-Priority Refresh Candidates: 16 pages
  • Total search impressions at risk: 196,678
  • Mean days un-updated: 205.8 days

Metadata/CTR Quick-Win Candidates: 12,114 pages
  • Total impressions available: 91,523,422
  • Mean position: 9.7 (Page 1/2 visibility)
  • Mean current CTR: 0.16%


## 3. Human review + the no-go list

### Required Human Review Checklist
Before any editorial modification is committed, a human editor must verify:
1. **Search Intent Check**: Has the primary SERP intent shifted from informational to commercial/tool-based?
2. **Factual & Temporal Currency**: Are pricing tables, year numbers, and software screenshots updated to current standards?
3. **Keyword Cannibalization**: Has a newer client page begun ranking for the same target query cluster?
4. **Technical Health**: Confirm the page is not experiencing 404/500 errors, canonical loops, or missing index tags.

---

### The No-Go List (Never Automate)
- 🚫 **Automated 301 Redirects / Deletions**: Never bulk-redirect or unpublish pages based purely on machine scores.
- 🚫 **Direct-to-CMS AI Auto-Publishing**: Do not bypass human editorial approval for content rewrites.
- 🚫 **YMYL Content Modifications**: Medical, legal, and financial advice articles must undergo strict domain-expert review.
- 🚫 **Core Commercial / High-Conversion Landing Pages**: Changes to checkout pages or primary lead-gen URLs require controlled A/B testing, not automated refresh.

In [3]:
# ---- TOP 10 PLAYBOOK RECOMMENDATIONS (HUMAN TRIAGE QUEUE) ----
cols_display = [
    "content_id", "client_id", "playbook_action", "reason_code",
    "predicted_decline_risk", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "is_declining_label"
]
top10_playbook = queue_df.head(10)[cols_display]

print("=== TOP 10 ACTION PLAYBOOK QUEUE ===")
print(top10_playbook.to_string())


=== TOP 10 ACTION PLAYBOOK QUEUE ===
                         content_id          client_id     playbook_action              reason_code  predicted_decline_risk  impressions_90d  days_since_last_update  avg_position   ctr  is_declining_label
playbook_rank                                                                                                                                                                                               
1              content_0a91db491d14  client_7f2253d7e2  REFRESH_AND_EXPAND  stale_high_volume_decay                0.954306            13299                     193          10.5  0.49                   1
2              content_1bfaa38ff26c  client_7f2253d7e2  REFRESH_AND_EXPAND  stale_high_volume_decay                0.931119            25715                     194          22.2  0.23                   1
3              content_fe16a55cd13d  client_7f2253d7e2  REFRESH_AND_EXPAND  stale_high_volume_decay                0.917421             4556   

## 4. Monitoring / retrain triggers

To prevent model decay in production, we define explicit monitoring signals and retrain triggers:

| Trigger Type | Metric / Signal | Threshold for Action | Required Action |
|---|---|---|---|
| **Performance Degradation** | Human Review Precision@20 | Drops below **70%** on weekly review batches | Audit recent false positives; recalibrate feature weights |
| **Data Distribution Drift** | Feature PSI (Population Stability Index) on `impressions_90d` or `avg_position` | $\text{PSI} > 0.20$ vs. training baseline | Trigger full model retraining with latest 90-day snapshot |
| **Calendar Cadence** | Scheduled monthly warehouse refresh | Every **30–45 days** | Ingest new monthly partition; execute automated validation suite |
| **Search Engine Core Update** | Industry-wide SERP volatility index | Volatility spike $> 2\times$ historical mean | Pause automated triage for 14 days; re-evaluate baseline after volatility settles |

In [4]:
# ---- SIMULATED RETRAIN & DRIFT CHECK MONITOR ----
print("=== AUTOMATED RETRAIN MONITOR STATUS ===")
monitor_checks = [
    {"Check": "Validation Precision@20", "Value": "90.0%", "Threshold": "> 70.0%", "Status": "HEALTHY"},
    {"Check": "Client Overlap in Eval", "Value": "0.0%", "Threshold": "== 0.0%", "Status": "HEALTHY"},
    {"Check": "Data Leakage Guard", "Value": "0 leaky features", "Threshold": "== 0", "Status": "HEALTHY"},
    {"Check": "Cadence Elapsed", "Value": "14 days", "Threshold": "< 45 days", "Status": "HEALTHY"},
]
print(pd.DataFrame(monitor_checks).to_string(index=False))


=== AUTOMATED RETRAIN MONITOR STATUS ===
                  Check            Value Threshold  Status
Validation Precision@20            90.0%   > 70.0% HEALTHY
 Client Overlap in Eval             0.0%   == 0.0% HEALTHY
     Data Leakage Guard 0 leaky features      == 0 HEALTHY
        Cadence Elapsed          14 days < 45 days HEALTHY


## 5. Exports for the paper

We export all artifacts necessary for the research paper and capstone report:
1. **`work/outputs/action_playbook_queue.csv`**: Full 30,000-item prioritized action queue.
2. **`work/outputs/playbook_metrics.json`**: Audited performance receipts (Precision@K, action breakdown, workload estimates).
3. **`work/figures/`**: Reusable publication-grade charts (Action distribution, Precision@K comparison curve, Feature importance).

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Ensure output directories exist
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export Action Queue CSV
queue_csv_path = "work/outputs/action_playbook_queue.csv"
queue_df.to_csv(queue_csv_path)
print(f"✓ Exported Action Playbook Queue: {queue_csv_path} ({len(queue_df):,} rows)")

# 2. Compute and Export Metrics JSON Receipts
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(topk.mean())

y_test_sub = y[test_idx]
test_scores = df.iloc[test_idx]["predicted_decline_risk"].values

# Baseline scores on test set
test_df_sub = df.iloc[test_idx]
is_stale_sub = (test_df_sub["days_since_last_update"] >= 180).astype(int)
is_vis_sub = (test_df_sub["impressions_90d"] >= 100).astype(int)
baseline_test_scores = (is_stale_sub * is_vis_sub * np.log2(1 + test_df_sub["impressions_90d"])).values

metrics_payload = {
    "task": "content_refresh_prioritisation",
    "evaluation_split": "grouped_by_client_id",
    "test_rows": int(len(test_idx)),
    "test_clients": int(len(set(groups[test_idx]))),
    "base_declining_rate": float(round(y_test_sub.mean(), 4)),
    "model_precision_at_10": precision_at_k(test_scores, y_test_sub, 10),
    "model_precision_at_20": precision_at_k(test_scores, y_test_sub, 20),
    "model_precision_at_50": precision_at_k(test_scores, y_test_sub, 50),
    "baseline_precision_at_20": precision_at_k(baseline_test_scores, y_test_sub, 20),
    "baseline_precision_at_50": precision_at_k(baseline_test_scores, y_test_sub, 50),
    "action_breakdown": df["playbook_action"].value_counts().to_dict(),
    "high_priority_refresh_count": int((df["playbook_action"] == "REFRESH_AND_EXPAND").sum()),
    "ctr_quickwin_count": int((df["playbook_action"] == "CTR_METADATA_OPTIMIZE").sum()),
}

metrics_json_path = "work/outputs/playbook_metrics.json"
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
print(f"✓ Exported Metrics JSON: {metrics_json_path}")

# 3. Export Publication-Ready Figures
# Figure 1: Playbook Action Distribution
plt.figure(figsize=(8, 4.5), dpi=300)
action_counts = df["playbook_action"].value_counts()
colors = ["#d95f02", "#7570b3", "#e7298a", "#e6ab02", "#1b9e77"]
bars = plt.barh(action_counts.index, action_counts.values, color=colors[:len(action_counts)])
plt.xlabel("Number of Content Items", fontsize=11)
plt.title("Content Portfolio Distribution Across Action Playbook Archetypes", fontsize=12, pad=12)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 200, bar.get_y() + bar.get_height()/2, f"{w:,} ({w/len(df):.1%})", va='center', fontsize=9)
plt.xlim(0, max(action_counts.values) * 1.2)
plt.tight_layout()
fig1_path = "work/figures/action_distribution.png"
plt.savefig(fig1_path)
plt.close()
print(f"✓ Exported Figure 1: {fig1_path}")

# Figure 2: Precision@K Comparison Curve
k_vals = [10, 20, 30, 50, 100, 200]
model_p_k = [precision_at_k(test_scores, y_test_sub, k) for k in k_vals]
base_p_k = [precision_at_k(baseline_test_scores, y_test_sub, k) for k in k_vals]
random_rate = [y_test_sub.mean()] * len(k_vals)

plt.figure(figsize=(7.5, 4.5), dpi=300)
plt.plot(k_vals, model_p_k, marker='o', color='#1b9e77', linewidth=2, label='HistGradientBoosting (Grouped Split)')
plt.plot(k_vals, base_p_k, marker='s', color='#d95f02', linewidth=2, linestyle='--', label='Week-4 Rule Baseline')
plt.axhline(y=y_test_sub.mean(), color='gray', linestyle=':', label=f'Random Base Rate ({y_test_sub.mean():.3f})')
plt.xlabel("Cutoff K (Top K Ranked Pages)", fontsize=11)
plt.ylabel("Precision@K (% Declining Pages)", fontsize=11)
plt.title("Precision@K vs. Baseline on Unseen Client Holdout", fontsize=12, pad=12)
plt.ylim(0.4, 1.05)
plt.legend(loc='upper right', frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig2_path = "work/figures/precision_at_k.png"
plt.savefig(fig2_path)
plt.close()
print(f"✓ Exported Figure 2: {fig2_path}")


✓ Exported Action Playbook Queue: work/outputs/action_playbook_queue.csv (30,000 rows)
✓ Exported Metrics JSON: work/outputs/playbook_metrics.json
✓ Exported Figure 1: work/figures/action_distribution.png
✓ Exported Figure 2: work/figures/precision_at_k.png


## Self-check

Before submitting, confirmed each line:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Ranked actions, reason codes, and archetype mappings defined
- [x] Intended use, limits, human review checklist, and no-go list articulated
- [x] Monitoring and retrain triggers specified
- [x] Exports written to `work/outputs/` and publication figures committed to `work/figures/`
- [x] Committed to my repo under `work/notebooks/w07_action_playbook.ipynb`.